# 02 Validation analysis

Reads selected completed OOF runs. Do not tune on Public LB.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

ROOT = Path('..') if not Path('artifacts').exists() and Path('../artifacts').exists() else Path('.')
RUNS = ROOT / 'artifacts' / 'runs'
FIG = ROOT / 'reports' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

# Edit this list to the selected completed run directories (folder names).
SELECTED = []
print('selected', SELECTED)


In [ ]:
rows = []
for run_id in SELECTED:
    run = RUNS / run_id
    metrics = json.loads((run / 'metrics.json').read_text())
    oof = pd.read_parquet(run / 'oof_predictions.parquet')
    auc = roc_auc_score(oof['addicted_label'], oof['prediction'])
    rows.append({'run_id': run_id, 'model': metrics.get('model_name'), 'oof_auc': auc, 'n': len(oof)})
summary = pd.DataFrame(rows)
display(summary)
if not summary.empty:
    summary.to_csv(FIG / 'validation_selected_runs.csv', index=False)


In [ ]:
# Fold / seed comparison when fold_metrics.csv exists
for run_id in SELECTED:
    path = RUNS / run_id / 'fold_metrics.csv'
    if not path.is_file():
        continue
    folds = pd.read_csv(path)
    display(run_id, folds.groupby('seed')['auc'].agg(['mean', 'std', 'min', 'max']))
    ax = folds.boxplot(column='auc', by='seed', figsize=(6, 4))
    plt.suptitle('')
    plt.title(f'{run_id} fold AUC by seed')
    plt.savefig(FIG / f'validation_folds_{run_id}.png', dpi=120)
    plt.close()


Public LB is a delayed / noisy signal. Weight search and Optuna budgets stay on local OOF only.
